# Gold Dim_aeroporto

`gold.dim_aeroporto` — dimensao de aeroporto, servindo origem E destino.

A dimensao nasce do FATO, nao do cadastro. Copiar `silver.aerodromos` daria 
496 linhas e deixaria 218 aeroportos do fato orfaos (os estrangeiros, que a 
ANAC nao cadastra). Uma dimensao existe para servir o fato: entao a lista de 
chaves vem do fato, e o cadastro ENRIQUECE por LEFT JOIN.

Regra de negocio que nasce aqui: a classificacao pais_aeroporto pelo prefixo
ICAO. Isso nao podia estar na silver — e interpretacao, nao aritmetica.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS voo_bem.gold

In [0]:
%sql
CREATE OR REPLACE TABLE voo_bem.gold.dim_aeroporto AS
WITH aeroportos_do_fato AS (
    SELECT DISTINCT icao_origem AS icao FROM voo_bem.silver.vra WHERE icao_origem IS NOT NULL AND icao_origem <> ''
    UNION
    SELECT DISTINCT icao_destino AS icao FROM voo_bem.silver.vra WHERE icao_destino IS NOT NULL AND icao_destino <> ''
),
cadastro AS (
    SELECT icao, nome, municipio, uf_nome, municipio_servido, uf_servido_nome
    FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY icao ORDER BY nome) AS rn
        FROM voo_bem.silver.aerodromos
        WHERE icao IS NOT NULL AND icao <> ''
    )
    WHERE rn = 1
)
SELECT 
    a.icao AS icao_aeroporto,
    -- fallback textual obrigatorio para a IA nao ler NULL
    COALESCE(c.nome, concat('AEROPORTO FORA DO CADASTRO ANAC (', a.icao, ')')) AS nome_aeroporto,
    c.municipio AS municipio_aeroporto,
    c.uf_nome AS uf_aeroporto,
    -- regra de negocio nova: identificacao de pais por interpretacao do prefixo ICAO
    CASE WHEN a.icao RLIKE '^S[BDIJNSW]' THEN 'Brasil' ELSE 'Exterior' END AS pais_aeroporto,
    (c.icao IS NOT NULL) AS no_cadastro_anac,
    current_timestamp() AS _processado_em
FROM aeroportos_do_fato a
LEFT JOIN cadastro c ON a.icao = c.icao;

In [0]:
%sql
SELECT * FROM voo_bem.gold.dim_aeroporto

# Gold Fatos_voos

`gold.fato_voos` — uma linha por etapa de voo.

E AQUI que nascem as regras de negócio que a silver não podia ter:

1. **PONTUALIDADE a 15 minutos** (`partida_pontual` / `chegada_pontual`). -- O número 15 é decisão de cliente. Na silver ele fecharia porta;
2. **ESCOPO doméstico / internacional**, a partir do tipo de linha. 
3. **As DECISÕES SOBRE A QUARENTENA**.

Companhia e códigos de operação entram como **DIMENSÃO DEGENERADA**: código e descrição no próprio fato, porque são poucos atributos, não mudam no tempo e o consumidor final é um LLM — cada *join* a menos é um erro a menos.

---

### DECISÕES SOBRE A QUARENTENA (213.543 registros diagnosticados)

* **a) AEROPORTO FORA DO CADASTRO DA ANAC (105.932) -> MANTIDO.** -- Não é dado inválido, é aeroporto estrangeiro. Descartar mataria a pergunta P4 do projeto.

In [0]:
%sql
CREATE OR REPLACE TABLE voo_bem.gold.fato_voos AS
WITH vra_sem_duplicata AS (
    SELECT * FROM (
        SELECT *,
            ROW_NUMBER() OVER (
                PARTITION BY icao_empresa, numero_voo, codigo_di,
                             codigo_tipo_linha,
                             icao_origem, icao_destino, partida_prevista,
                             partida_real,
                             chegada_prevista, chegada_real, situacao_voo
                ORDER BY __ingerido_em
            ) AS rn
        FROM voo_bem.silver.vra
    )
    WHERE rn = 1
),
empresa AS (
    SELECT icao, razao_social, origem_cadastro
    FROM (
        SELECT *, ROW_NUMBER() OVER (
            PARTITION BY icao
            ORDER BY CASE WHEN situacao = 'ATIVA' THEN 0 ELSE 1 END, razao_social
        ) AS rn
        FROM voo_bem.silver.empresas
        WHERE icao IS NOT NULL AND icao <> ''
    )
    WHERE rn = 1
),
di AS (
    SELECT codigo, descricao FROM voo_bem.silver.codigos_operacao WHERE dominio = 'codigo_di'
),
tipo_linha AS (
    SELECT codigo, descricao FROM voo_bem.silver.codigos_operacao WHERE dominio = 'codigo_tipo_linha'
),
base AS (
    SELECT 
        v.*,
        -- decisao (c): metrica fora da faixa plausivel vira NULL
        (v.atraso_partida_min IS NOT NULL AND (v.atraso_partida_min < -120 OR v.atraso_partida_min > 1440))
        OR (v.atraso_chegada_min IS NOT NULL AND (v.atraso_chegada_min < -120 OR v.atraso_chegada_min > 1440)) AS atraso_fora_de_faixa,
        
        CASE 
            WHEN v.atraso_partida_min IS NOT NULL AND (v.atraso_partida_min < -120 OR v.atraso_partida_min > 1440) THEN NULL 
            ELSE v.atraso_partida_min 
        END AS atraso_partida_min_tratado,
        
        CASE 
            WHEN v.atraso_chegada_min IS NOT NULL AND (v.atraso_chegada_min < -120 OR v.atraso_chegada_min > 1440) THEN NULL 
            ELSE v.atraso_chegada_min 
        END AS atraso_chegada_min_tratado
    FROM vra_sem_duplicata v
)
SELECT 
    -- ===== dimensao degenerada: companhia =====
    b.icao_empresa,
    COALESCE(e.razao_social, concat('COMPANHIA NAO CADASTRADA (', b.icao_empresa, ')')) AS nome_companhia,
    e.origem_cadastro AS cadastro_companhia,
    b.numero_voo,

    -- ===== dimensoes degeneradas: codigos de operacao =====
    b.codigo_di,
    COALESCE(d.descricao, concat('Codigo nao catalogado (', b.codigo_di, ')')) AS descricao_di,
    
    b.codigo_tipo_linha,
    COALESCE(t.descricao, concat('Codigo nao catalogado (', b.codigo_tipo_linha, ')')) AS descricao_tipo_linha,

    -- ===== escopo do voo =====
    CASE 
        WHEN b.codigo_tipo_linha IN ('N', 'C') THEN 'Domestico'
        WHEN b.codigo_tipo_linha IN ('I', 'G') THEN 'Internacional'
        ELSE 'Nao classificado'
    END AS escopo_voo,

    -- ===== chaves para dim_aeroporto =====
    b.icao_origem,
    b.icao_destino,
    concat(b.icao_origem, ' - ', b.icao_destino) AS rota,

    -- ===== tempo =====
    b.partida_prevista,
    b.partida_prevista_data,
    b.partida_prevista_hora,
    hour(b.partida_prevista) AS hora_partida_prevista,
    CASE dayofweek(b.partida_prevista_data)
        WHEN 1 THEN 'domingo' WHEN 2 THEN 'segunda' WHEN 3 THEN 'terca'
        WHEN 4 THEN 'quarta' WHEN 5 THEN 'quinta' WHEN 6 THEN 'sexta'
        WHEN 7 THEN 'sabado'
    END AS dia_semana,
    date_trunc('MONTH', b.partida_prevista_data) AS mes_referencia,
    b.partida_real,
    b.chegada_prevista,
    b.chegada_real,

    -- ===== metricas (decisao c aplicada) =====
    CASE WHEN b.atraso_fora_de_faixa THEN NULL ELSE b.atraso_partida_min END AS atraso_partida_min,
    CASE WHEN b.atraso_fora_de_faixa THEN NULL ELSE b.atraso_chegada_min END AS atraso_chegada_min,
    CASE WHEN b.atraso_fora_de_faixa THEN NULL ELSE b.minutos_recuperados END AS minutos_recuperados,
    b.atraso_fora_de_faixa,

    -- ===== REGRA DE NEGOCIO: pontualidade a 15 minutos =====
    CASE WHEN b.atraso_fora_de_faixa OR b.atraso_partida_min IS NOT NULL THEN NULL
         ELSE b.atraso_partida_min <= 15 END AS partida_pontual,
    CASE WHEN b.atraso_fora_de_faixa OR b.atraso_chegada_min IS NOT NULL THEN NULL
         ELSE b.atraso_chegada_min <= 15 END AS chegada_pontual,

    -- ===== situacao =====
    b.situacao_voo,
    (b.situacao_voo = 'CANCELADO') AS voo_cancelado,
    (b.situacao_voo = 'REALIZADO') AS voo_realizado,

    -- ===== controle =====
    current_timestamp() AS _processado_em

FROM base b
LEFT JOIN empresa e ON b.icao_empresa = e.icao
LEFT JOIN di d ON b.codigo_di = d.codigo
LEFT JOIN tipo_linha t ON b.codigo_tipo_linha = t.codigo;

# Gold Obt_voos

`gold.obt_voos` — One Big Table, desenhada para um consumidor específico: uma IA.

Uma linha por etapa de voo, com TUDO resolvido: nome de companhia, nome de aeroporto de origem e destino com município e UF, tipo de linha por extenso, escopo, pontualidade e as métricas de atraso prontas.

Regra de ouro desta tabela: nenhuma coluna de código sem a coluna de descrição correspondente ao lado. O LLM lê nome, não código ICAO.

E o unico join que ele exige de quem consome: nenhum 

In [0]:
%sql
CREATE OR REPLACE TABLE voo_bem.gold.obt_voos AS
SELECT 
    -- ===== companhia =====
    f.icao_empresa,
    f.nome_companhia,
    f.numero_voo,

    -- ===== operacao =====
    f.codigo_di,
    f.descricao_di,
    f.codigo_tipo_linha,
    f.descricao_tipo_linha,
    f.escopo_voo,

    -- ===== origem =====
    f.icao_origem,
    o.nome_aeroporto AS nome_aeroporto_origem,
    o.municipio_aeroporto AS municipio_origem,
    o.uf_aeroporto AS uf_origem,
    o.pais_aeroporto AS pais_origem,

    -- ===== destino =====
    f.icao_destino,
    d.nome_aeroporto AS nome_aeroporto_destino,
    d.municipio_aeroporto AS municipio_destino,
    d.uf_aeroporto AS uf_destino,
    d.pais_aeroporto AS pais_destino,

    -- ===== rota, em codigo e por extenso =====
    f.rota AS rota_icao,
    concat(
        coalesce(o.municipio_aeroporto, f.icao_origem), 
        ' - ', 
        coalesce(d.municipio_aeroporto, f.icao_destino)
    ) AS rota_municipios,

    -- ===== tempo =====
    f.partida_prevista,
    f.partida_prevista_data,
    f.partida_prevista_hora,
    f.hora_partida_prevista,
    f.dia_semana,
    f.mes_referencia,
    f.partida_real,
    f.chegada_prevista,
    f.chegada_real,

    -- ===== metricas =====
    f.atraso_partida_min,
    f.atraso_chegada_min,
    f.minutos_recuperados,
    f.atraso_fora_de_faixa,
    f.partida_pontual,
    f.chegada_pontual,

    -- ===== situacao =====
    f.situacao_voo,
    f.voo_realizado,
    f.voo_cancelado,

    -- ===== controle =====
    f._processado_em

FROM voo_bem.gold.fato_voos f
LEFT JOIN voo_bem.gold.dim_aeroporto o ON f.icao_origem = o.icao_aeroporto
LEFT JOIN voo_bem.gold.dim_aeroporto d ON f.icao_destino = d.icao_aeroporto;